# 外部 HTTP 服务调用

学习目标：选择并复用 HTTPX 客户端，区分超时与上游错误，限制并发请求，并在取消或失败后关闭连接资源。

前置知识：HTTP 客户端、协程、异常处理、并发控制与资源生命周期。

适用条件：Python 3.12、HTTPX 0.28.1；本地上游使用课程环境中的 FastAPI 与 Uvicorn。全部输入都在本地，不需要账号、外网 API 或付费服务。

环境准备：[FastAPI 环境与运行说明](README.md)。选择课程环境的 Python 3 (ipykernel)，从空内核顺序执行。Notebook 和终端工作目录均为 content/Web与应用开发/FastAPI。第 4 节开始使用端口 8170，按该节说明启动服务，最后关闭。

配套脚本：位于 scripts/17-external-http/。

1. [upstream.py](scripts/17-external-http/upstream.py)：提供本章讲解的成功、500 错误和延迟文本接口，供客户端做真实网络实验。

## 1 先完成一次同步调用

应用调用的另一个 HTTP 服务通常称为“上游服务”。HTTPX 的 Client 提供同步请求方法，get 返回后再处理响应。先用 MockTransport 把请求交给一个本地函数，专注观察 URL、状态码和 JSON。

MockTransport 不进行真实网络请求，下面也没有启动服务器；模拟响应不能用于证明 TCP 连接复用或网络超时。

In [1]:
import httpx


def mock_record(request: httpx.Request) -> httpx.Response:
    return httpx.Response(200, json={"path": request.url.path, "title": "学习 HTTP"})


with httpx.Client(
    transport=httpx.MockTransport(mock_record), base_url="http://upstream.test"
) as client:
    response = client.get("/records/1")
    response.raise_for_status()

assert response.json()["path"] == "/records/1"
assert client.is_closed
print(response.status_code, response.json())
print("客户端已关闭：", client.is_closed)

200 {'path': '/records/1', 'title': '学习 HTTP'}
客户端已关闭： True


## 2 在异步调用中复用客户端

AsyncClient 的请求方法需要 await，适合异步函数中的网络调用。同步 Client 的真实网络调用会阻塞当前调用线程，不能仅因为放进 async def 就变成异步 I/O。

一个客户端可用于一组请求，并共享 base_url 等配置。真实网络中，客户端的连接池可以复用连接；频繁在循环内部新建客户端会失去这种复用机会。用 async with 明确客户端的使用范围，退出时关闭连接；需要显式关闭时使用 await client.aclose()。

这里继续使用前面定义的 mock_record，只对照异步写法。请求在循环中逐个 await，仍是顺序发送。

In [2]:
async with httpx.AsyncClient(
    transport=httpx.MockTransport(mock_record), base_url="http://upstream.test"
) as client:
    paths = []
    for record_id in [1, 2]:
        response = await client.get(f"/records/{record_id}")
        response.raise_for_status()
        paths.append(response.json()["path"])

assert paths == ["/records/1", "/records/2"]
assert client.is_closed
print(paths)
print("异步客户端已关闭：", client.is_closed)

['/records/1', '/records/2']
异步客户端已关闭： True


## 3 分开配置四种超时

HTTPX 把等待分成四类，配置值的单位均为秒。它们限制各自阶段的等待，并不等于整个业务操作的总时限。

| 参数 | 中文名称／含义 | 对应异常 |
| --- | --- | --- |
| connect | 连接超时：等待建立套接字连接 | ConnectTimeout |
| read | 读取超时：等待收到一块数据 | ReadTimeout |
| write | 写入超时：等待发送一块数据 | WriteTimeout |
| pool | 连接池超时：等待获得池中的连接 | PoolTimeout |

例如，对方已经发来一部分响应，后续数据一直未到，也可能触发读取超时。以下只是构造配置，真正的读超时和连接池超时在本地服务运行后观察；本章不人为制造连接和写入阶段的网络故障。

In [3]:
timeout = httpx.Timeout(connect=2.0, read=0.2, write=2.0, pool=0.2)
limits = httpx.Limits(max_connections=2, max_keepalive_connections=2)

# 四项均显式设置；Limits 单独限制连接数量，单位不是秒。
assert timeout.read == 0.2 and limits.max_connections == 2
print(timeout)
print("连接数上限：", limits.max_connections)

Timeout(connect=2.0, read=0.2, write=2.0, pool=0.2)
连接数上限： 2


## 4 启动一个有限的本地上游

现在需要真实端口。上游先提供一个记录查询，delay 是 0 到 1 秒的有限等待。响应里的 peer_port 是服务端看到的客户端端口，仅用于本次连接观察。下面定义应用和路由，不在 Notebook 中启动服务器。

In [4]:
import asyncio
from typing import Annotated

from fastapi import FastAPI, HTTPException, Query, Request

upstream = FastAPI(title="本地 HTTP 上游")

@upstream.get("/records/{record_id}")
async def read_record(
    record_id: int,
    request: Request,
    delay: Annotated[float, Query(ge=0, le=1)] = 0,
) -> dict[str, int | str | None]:
    await asyncio.sleep(delay)
    return {
        "id": record_id,
        "title": f"学习记录 {record_id}",
        "peer_port": request.client.port if request.client else None,
    }

再增加两种可控情况：一个接口直接返回 500；另一个接口先发出 ready 文本，等待 1 秒再发出 done。StreamingResponse 在这里仅用于分两次提供响应内容，使客户端能观察等待和取消。生成器的 finally 在退出时输出一行关闭信息。

In [5]:
from collections.abc import AsyncIterator

from fastapi.responses import StreamingResponse

@upstream.get("/failure")
def fail() -> None:
    raise HTTPException(500, "上游示例故意返回错误")


async def delayed_lines() -> AsyncIterator[bytes]:
    try:
        yield b"ready\n"
        await asyncio.sleep(1.0)
        yield b"done\n"
    finally:
        print("slow_text: generator closed", flush=True)


@upstream.get("/slow-text")
def slow_text() -> StreamingResponse:
    return StreamingResponse(delayed_lines(), media_type="text/plain")

配套 upstream.py 只组合上述三条路由，导入模块不会启动服务。为了让本地调用不受代理环境变量影响，后面的客户端设置 trust_env=False。

Step 1：在课程目录的终端启动上游服务。

```powershell
python -m uvicorn upstream:upstream --app-dir scripts/17-external-http --host 127.0.0.1 --port 8170
```

Step 2：等待终端显示应用启动完成，再执行下面及后续代码单元。

同一个 Client 顺序请求同一上游时，可以观察本次 HTTP/1.1 连接是否保留。连接是否仍可复用也取决于服务端是否关闭连接；这里不测性能或预设加速倍数。

In [6]:
base_url = "http://127.0.0.1:8170"
with httpx.Client(base_url=base_url, timeout=2.0, trust_env=False) as client:
    records = []
    for record_id in [1, 2]:
        response = client.get(f"/records/{record_id}")
        response.raise_for_status()
        records.append(response.json())

ports = [record["peer_port"] for record in records]
assert [record["id"] for record in records] == [1, 2]
assert ports[0] is not None and ports[0] == ports[1]
assert client.is_closed
# 本次相同的对端端口与顺序请求一起支持连接复用的观察。
print("记录编号：", [record["id"] for record in records])
print("两次使用相同客户端端口：", ports[0] == ports[1])
print("客户端已关闭：", client.is_closed)

记录编号： [1, 2]
两次使用相同客户端端口： True
客户端已关闭： True


## 5 区分上游状态错误和传输超时

收到 500 说明已经拿到了 HTTP 响应。get 本身不会仅因这个状态码抛异常，调用 raise_for_status 后才以 HTTPStatusError 处理；异常的 response 可用于读取状态。

连接失败或超时则属于 RequestError 这一类请求异常，TimeoutException 是其中的超时基类。按需要捕获具体类型，避免把状态码错误、网络问题和自己的程序错误全部合并成一条空结果。

In [7]:
async with httpx.AsyncClient(
    base_url=base_url, timeout=2.0, trust_env=False
) as client:
    response = await client.get("/failure")
    assert response.status_code == 500
    try:
        response.raise_for_status()
    except httpx.HTTPStatusError as error:
        print(type(error).__name__, error.response.status_code)
    else:
        raise AssertionError("500 应在 raise_for_status 时抛出异常")

assert client.is_closed

HTTPStatusError 500


上游的两段文本间隔 1 秒，超过 read=0.2 秒，因此普通 get 在读取完整响应时会失败。这个实验连接已建立，也已经收到第一段数据，检查的是后续读取等待。

In [8]:
async with httpx.AsyncClient(
    base_url=base_url, timeout=timeout, trust_env=False
) as client:
    try:
        await client.get("/slow-text")
    except httpx.ReadTimeout as error:
        print(type(error).__name__, error.request.url.path)
    else:
        raise AssertionError("两段数据的间隔应触发读取超时")

assert client.is_closed
print("超时后客户端已关闭：", client.is_closed)

ReadTimeout /slow-text
超时后客户端已关闭： True


## 6 连接池数量与并发数量

max_connections 限制连接池能创建的连接数，max_keepalive_connections 限制保留的空闲连接数；它们不等于应用创建的任务数。等待可用连接的时间由 pool 控制。

先把连接池限制为一个连接。stream 上下文会在收到响应头后交给调用方逐步读取；aiter_lines 用于逐行读取。先拿到 ready，确认首段已收到，而第二段尚未读完时连接仍被占用，第二个请求只能等待。退出 stream 上下文会关闭响应并释放占用。

In [9]:
one_connection = httpx.Limits(max_connections=1, max_keepalive_connections=1)
pool_timeout = httpx.Timeout(2.0, pool=0.1)
async with httpx.AsyncClient(
    base_url=base_url, limits=one_connection, timeout=pool_timeout, trust_env=False
) as client:
    async with client.stream("GET", "/slow-text") as held:
        held.raise_for_status()
        lines = held.aiter_lines()
        assert await anext(lines) == "ready"
        try:
            await client.get("/records/1")
        except httpx.PoolTimeout:
            print("PoolTimeout：连接仍被前一个响应占用")
        else:
            raise AssertionError("只有一个连接且被占用时应等待超时")
    assert held.is_closed
    recovered = await client.get("/records/1")
    recovered.raise_for_status()
    print("关闭前一个响应后：", recovered.status_code)

assert client.is_closed

PoolTimeout：连接仍被前一个响应占用
关闭前一个响应后： 200


应用还可以用 Semaphore 限制进入请求操作的任务数量。下面的 batch 一次接收四个编号，只允许两个任务进入请求区间。计数记录的是客户端正在执行这段操作的数量，不是服务端的 worker 数量。

同一个 AsyncClient 交给全部任务使用。async with gate 退出时释放名额；TaskGroup 会等待组内任务结束，若某个任务失败，也会取消并等待其余任务。TaskGroup 从 Python 3.11 提供，本章使用 Python 3.12。

In [10]:
async def batch(record_ids: list[int]) -> tuple[list[int], int, bool]:
    gate = asyncio.Semaphore(2)
    active = 0
    peak = 0
    async with httpx.AsyncClient(
        base_url=base_url, limits=limits, timeout=2.0, trust_env=False
    ) as client:
        async def fetch(record_id: int) -> int:
            nonlocal active, peak
            async with gate:
                active += 1
                peak = max(peak, active)
                try:
                    response = await client.get(f"/records/{record_id}", params={"delay": 0.1})
                    response.raise_for_status()
                    return response.json()["id"]
                finally:
                    active -= 1

        async with asyncio.TaskGroup() as group:
            tasks = [group.create_task(fetch(record_id)) for record_id in record_ids]
    assert active == 0
    return [task.result() for task in tasks], peak, client.is_closed


ids, peak, closed = await batch([1, 2, 3, 4])
assert ids == [1, 2, 3, 4] and 1 <= peak <= 2 and closed
print("返回编号：", ids, "请求区间峰值：", peak, "客户端已关闭：", closed)

返回编号： [1, 2, 3, 4] 请求区间峰值： 2 客户端已关闭： True


## 7 取消之后仍要等待清理

Task.cancel 发出取消请求，任务在可取消的位置收到 CancelledError。任务里的 finally 仍用于清理；不要在工作协程中把取消当作普通成功吞掉。发起取消的一方应继续 await 任务，等清理结束。

为了确认取消发生在真实响应读取期间，下面先等待上游发来 ready。Event 用于通知“第一行已经收到”；aiter_lines 逐行异步读取响应。stream 上下文退出时关闭响应，外层 finally 记录关闭状态。

In [11]:
started = asyncio.Event()
cleanup = {}


async def consume_slow(client: httpx.AsyncClient) -> None:
    response = None
    try:
        async with client.stream("GET", "/slow-text") as response:
            response.raise_for_status()
            lines = response.aiter_lines()
            first = await anext(lines)
            assert first == "ready"
            started.set()
            await anext(lines)
    finally:
        cleanup["response_closed"] = response is not None and response.is_closed

接着创建任务，等确认首行已到后取消。wait_for 的 3 秒限制用于防止实验一直等不到通知。无论等待是否成功，finally 都取消并等待任务，然后才退出客户端上下文。

In [12]:
async with httpx.AsyncClient(
    base_url=base_url, timeout=2.0, trust_env=False
) as client:
    task = asyncio.create_task(consume_slow(client))
    try:
        await asyncio.wait_for(started.wait(), timeout=3.0)
    finally:
        task.cancel()
        try:
            await task
        except asyncio.CancelledError:
            # 此处是发起取消的一方，接收任务已经取消的确认。
            print("读取任务已取消并完成清理")

assert task.cancelled() and cleanup["response_closed"] and client.is_closed
print("响应已关闭：", cleanup["response_closed"])
print("客户端已关闭：", client.is_closed)

读取任务已取消并完成清理
响应已关闭： True
客户端已关闭： True


## 8 给一组异步操作设置总时限

如果希望连同应用排队一起限制等待，可以在操作外再使用 asyncio.timeout。它从 Python 3.11 提供，通过取消当前任务中断超时等待，并在上下文外转换为 TimeoutError；与 HTTPX 的 ReadTimeout 是不同层次的约束。

这里允许单次读取等 2 秒，但整个调用最多等 0.2 秒。客户端放在总时限范围内，退出时仍执行关闭。取消客户端等待并不承诺远程业务已回滚；本例上游只发送有限文本，不执行写入。

In [13]:
try:
    async with asyncio.timeout(0.2):
        async with httpx.AsyncClient(
            base_url=base_url, timeout=2.0, trust_env=False
        ) as client:
            await client.get("/slow-text")
except TimeoutError:
    print("TimeoutError：本次整体等待达到时限")
else:
    raise AssertionError("整体时限应早于第二段文本到达")

assert client.is_closed
print("总时限到达后客户端已关闭：", client.is_closed)

TimeoutError：本次整体等待达到时限
总时限到达后客户端已关闭： True


客户端结束不代表上游进程已退出。完成实验后关闭独立服务。

Step 1：回到上游服务终端，按 Ctrl+C，等待应用关闭完成。

Step 2：确认终端已返回提示符，且 http://127.0.0.1:8170/records/1 已不能访问。

## 本章小结

（1）同步 Client 和异步 AsyncClient 使用不同调用方式；客户端应覆盖一组相关请求的使用范围，并在结束时关闭。

（2）MockTransport 用于可控响应测试；连接复用、读取超时和连接池等待需要在真实网络条件下观察。

（3）四种 HTTPX 超时分别限制建立连接、读取、写入和等待池连接。HTTPStatusError 表示已收到状态错误的响应，与传输异常不同。

（4）连接数、应用并发数和总等待时限各有职责。取消后等待任务清理，关闭流式响应和客户端，最后退出本地上游服务。

自查：为什么 async with 放进每次循环并不能帮助连接复用？为什么 pool 超时不代表上游已经执行了请求？

## 练习

1. 给 mock_record 增加 /missing 分支并返回 404。在客户端调用 raise_for_status，确认捕获的是 HTTPStatusError，且异常中的 response.status_code 为 404。
2. 把真实 /slow-text 调用的 read 超时改为 2 秒。确认响应包含 ready 和 done 两行，再恢复短读取时限并确认 ReadTimeout；不要用固定运行耗时作断言。
3. 将 batch 的 Semaphore 改为 1，仍请求四个编号。确认编号齐全、请求区间峰值为 1、客户端已关闭；连接池上限保留 2，说明两个上限的区别。
4. 在取消示例中记录是否读到第二行。保持首行到达后立即取消，检查任务取消、响应关闭、客户端关闭，且没有把未完成的第二行当作成功结果。

### 提示

- 第 1 题只改模拟响应，不需要真实服务；断言错误类型和状态。
- 第 2 题普通 get 会读取完整响应，成功时可用 response.text.splitlines() 检查内容。
- 第 3 题观察 peak 计数，不比较固定加速倍数。
- 第 4 题保留 finally 与 await task，记录值只能在确实读到数据之后修改。

## 参考与引用来源

- **HTTPX 官方文档**：[Clients](https://www.python-httpx.org/advanced/clients/)，定位 Why use a Client、Usage 与 Sharing configuration，支持连接池复用、上下文关闭与公共配置；[Async Support](https://www.python-httpx.org/async/)，定位 Making Async requests、Opening and closing clients 与 Streaming responses，支持 await、复用范围、stream、aiter_lines 和响应关闭；[Mock transports](https://www.python-httpx.org/advanced/transports/#mock-transports)，支持不经过真实网络的请求替身；[Timeouts](https://www.python-httpx.org/advanced/timeouts/#fine-tuning-the-configuration)，定位四种超时的不同等待阶段；[Resource Limits](https://www.python-httpx.org/advanced/resource-limits/)，支持连接与空闲连接上限；[Exceptions](https://www.python-httpx.org/exceptions/)，定位 RequestError、HTTPStatusError 和四个超时异常；[Environment Variables](https://www.python-httpx.org/environment_variables/)，支持 trust_env=False 不读取环境代理等配置。
- **Python 3.12 官方文档**：[Coroutines and Tasks](https://docs.python.org/3.12/library/asyncio-task.html)，定位 Creating Tasks、Task Cancellation、Task Groups、Timeouts、Task.cancel，支持任务等待、取消传播、finally 清理、TaskGroup 与 asyncio.timeout 的版本和行为；[Synchronization Primitives](https://docs.python.org/3.12/library/asyncio-sync.html)，定位 Event 与 Semaphore，支持首行通知、限制进入请求区间的任务数与退出释放。
- **FastAPI 官方文档**：[Using the Request Directly](https://fastapi.tiangolo.com/advanced/using-request-directly/)，支持在路由中读取 Request；[StreamingResponse](https://fastapi.tiangolo.com/advanced/custom-response/#streamingresponse)，支持异步生成器逐块响应；[Handling Errors](https://fastapi.tiangolo.com/tutorial/handling-errors/#use-httpexception)，支持故意返回的 500 上游示例；[Numeric Validations](https://fastapi.tiangolo.com/tutorial/path-params-numeric-validations/)，定位 Number validations 与 Recap，支持 Query 的 ge、le 数值约束。
- **Starlette 官方文档**：[Client Address](https://starlette.dev/requests/#client-address)，支持 request.client 的 host、port 及可能为 None 的条件。
- **Uvicorn 官方文档**：[Settings](https://uvicorn.dev/settings/)，定位 Application 与 Socket Binding，支持导入字符串、app-dir、本地监听地址与端口设置。